# The replication (Study 1 extension)

RQ1's main hypothesis re-tested without the index-6/7 leak, at completely new seeds
(gp_reward-priors `HANDOFF_HP_SELECTION.md` §14.83):

- new reward models for MR, PT and NMR, selected at seed 100 (patience 15, §14.146–§14.149) and
  trained at seeds 100-110 (§14.154); the ensemble mean reads MR's models;
- `return_range_split "length"` in both protocols;
- IQL at seeds 101-110, shown here as units 1-10, at the rule the normalization test adopts;
- the main hypothesis only: no ablations, companions or sensitivity arm.

The registered reading is `results/repl_readout.py rq1`: `rq1_readout`'s own functions on the
replication's cells.

**Gate (§14.5, as in Study 1):** a family's candidate (no-terminals) ladder cells are drawn only
after its registered reading on that ladder (`repl_readout.REGISTERED_READ`, the one record of
the replication's readings). Standard-protocol cells show their own mean ± sd when complete.
Descriptive figures compare within each reward only.

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb

HERE = os.path.abspath(".")
ROOT = os.path.dirname(HERE)
sys.path.insert(0, HERE)
sys.path.insert(0, ROOT)
import eval_readout as R  # noqa: E402
import exp_readout as X  # noqa: E402
import repl_readout as RP  # noqa: E402  the replication's collector, arrangement and gate
import rq1_readout as Q  # noqa: E402  the registered machinery

_argv, sys.argv = sys.argv, [sys.argv[0]]
import phase2_sweeps as P  # noqa: E402
import replication as Y  # noqa: E402

sys.argv = _argv

SCALE = 100.0
api = wandb.Api(timeout=120)
METHODS = [("MR", "MR"), ("PT", "PT"), ("NMR", "NMR"), ("ENS_MEAN", "ENS w/ MEAN")]
_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
COLORS = {m: _cycle[j + 1] for j, (m, _) in enumerate(METHODS)}  # C0 is the task reward, as Study 1
COLORS["task_reward"] = _cycle[0]
LABELS = dict(METHODS)
LABELS["task_reward"] = "Task reward (full data)"
GATED_CELL = {"gated": True, "audit": [], "incomplete": ["§14.5 gate"], "scores": {}}

## Loading

Every replication cell through `repl_readout.collect` (each cell at its expected index, scores
re-keyed seed s → unit s − 100), arranged in `rq1_readout`'s shapes. The seed-100 grids through
`replication.find_grids`.

In [ ]:
rcells, notes = RP.collect(api, P, Y)
for n in notes:
    print("note:", n)
std, cand, full, ctrl = RP.arrange(rcells, Y)
sp, E = P.spec(), P._conditions()
gpath = os.path.join(ROOT, Y.WINNERS)
gwin = json.load(open(gpath)) if os.path.exists(gpath) else {}
grids = Y.find_grids(api, sp, E)
gspec = Y.grid_spec(sp, E)
cspec = Y.cell_spec(sp, E)
print(f"grids: {sum(n in grids for n in gspec)}/{len(gspec)} launched, {sum(n in gwin for n in gspec)} winners written; "
      f"cells: {len(rcells)}/{len(cspec)} launched")

## Progress (candidate cells: states only)

Counts of cells by protocol and level: complete, incomplete, or not launched.

In [ ]:
def state(c):
    if c is None:
        return "not launched"
    return "complete" if R.cell_ready(c) is None else "incomplete"


rows = []
for prot in Y.PROTOCOLS:
    for lv in ("full",) + tuple(X.LEVELS) + tuple(X.NOISE_LEVELS):
        names = [n for n, c in cspec.items() if c["prot"] == prot and c.get("level") == lv]
        st = pd.Series([state(rcells.get(n)) for n in names]).value_counts()
        rows.append({"protocol": prot, "level": lv, "cells": len(names), **st.to_dict()})
display(pd.DataFrame(rows).fillna(0).set_index(["protocol", "level"]))

## Reward-model accuracy A (seeds 101-110)

The quantity A of RQ1's manipulation check: per-pair accuracy on each seed's clean, tie-free
validation split, from `exp/rq1_accuracy_repl.json` (measured on the box by
`rq1_accuracy.py measure --repl`).
- **Gate:** every MR, PT and NMR record must pass `rq1_readout`'s cross-check against the training
  run's logged `eval_acc_reloaded`, and a point is drawn only when all ten seeds are measured.
- The path-length floor is drawn only where the seeds' validation splits are on this machine
  (`gp_reward-priors/data/antmaze/<env>/eval/seed_<s>/`).

In [ ]:
sys.path.insert(0, os.path.join(ROOT, "gp_reward-priors", "scripts_bnn"))
import heuristic_floor as HF  # noqa: E402

acc_raw, anotes = Q.load_accuracy(RP.ACC_PATH, Q.logged_accuracies() if os.path.exists(RP.ACC_PATH) else None)
for n in anotes:
    print("note:", n)
acc = RP.rekey_accuracy(acc_raw)  # units 1-10


def load_floor():
    out = {}
    for v in X.VARIANTS:
        try:
            out[v] = [float(HF.floor_correct(HF.val_file(v.replace("-", "_"), 100 + s)).mean()) for s in X.SEEDS]
        except (OSError, FileNotFoundError, KeyError):
            pass
    return out


floor = load_floor()
print(f"path-length floor available for: {', '.join(floor) or 'none (validation splits not on this machine)'}")


def x_positions(axis, variant):
    if axis == "sparsity":
        levels = (X.FULL,) + X.AXES[axis]["levels"][::-1]
        lab = lambda lv: f"full\n({X.TRAIN_POOL[variant]})" if lv == X.FULL else str(X.SPARSE_PAIRS[lv])
        return {lv: (float(i), lab(lv)) for i, lv in enumerate(levels)}
    levels = (X.FULL,) + X.AXES[axis]["levels"]
    return {lv: (X._x_noise(variant, lv), "0\n(full)" if lv == X.FULL else f"{X.FLIP[lv]:g}") for lv in levels}


def acc_point(m, v, lv):
    vals = [acc.get((m, v, lv, s)) for s in X.SEEDS]
    bad = [x for x in vals if not isinstance(x, float)]
    if bad:
        return None, f"{10 - len(bad)}/10 measured"
    x = SCALE * np.array(vals)
    return (float(x.mean()), float(x.std(ddof=1))), None


def plot_acc(axis):
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    series = [m for m, _ in METHODS]
    for ax, v in zip(axes.ravel(), X.VARIANTS):
        pos = x_positions(axis, v)
        order = sorted(pos, key=lambda lv: pos[lv][0])
        xs_sorted = [pos[lv][0] for lv in order]
        step = 0.5 * min(b - a for a, b in zip(xs_sorted, xs_sorted[1:])) / len(series)
        for j, m in enumerate(series):
            dx = (j - (len(series) - 1) / 2) * step
            pts = [acc_point(m, v, lv)[0] for lv in order]
            xs = [pos[lv][0] + dx for lv in order]
            ys = [np.nan if p is None else p[0] for p in pts]
            es = [np.nan if p is None else p[1] for p in pts]
            if all(np.isnan(ys)):
                continue
            ax.plot(xs, ys, "-o", color=COLORS[m], lw=1.4, ms=5, label=LABELS[m])
            ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor=COLORS[m], capsize=3, lw=1)
        if v in floor:
            f = SCALE * np.array(floor[v])
            ax.axhline(f.mean(), ls="--", color="0.35", lw=1.2, label="path-length floor")
        ax.set_xticks([pos[lv][0] for lv in order])
        ax.set_xticklabels([pos[lv][1] for lv in order])
        ax.set_xlabel("preference pairs" if axis == "sparsity" else "flip fraction p")
        ax.set_ylabel("validation accuracy (per pair, × 100)")
        ax.set_title(f"antmaze-{v}")
        ax.grid(alpha=0.3)
    handles = {}
    for ax in axes.ravel():
        for h, lab in zip(*ax.get_legend_handles_labels()):
            handles.setdefault(lab, h)
    fig.legend(list(handles.values()), list(handles), loc="lower center", ncol=len(handles), frameon=False)
    fig.suptitle(f"Replication: reward-model accuracy, {axis} (mean ± sd over seeds 101-110)")
    fig.tight_layout(rect=(0, 0.05, 1, 0.96))
    plt.show()


plot_acc("sparsity")
plot_acc("noise")

## The ladders

Each reward's IQL score along the ladder, mean ± sd over seeds 101-110 (units 1-10).
**With terminals:** each cell's own mean ± sd when complete. **Without terminals:** only the
families in `repl_readout.REGISTERED_READ` (none until each family's replication reading is
made); the dashed line is the task reward at full data without terminals. Every point is
complete and audit-clean.

In [ ]:
def stats(cell):
    x = SCALE * np.array([cell["scores"][s] for s in X.SEEDS])
    return float(x.mean()), float(x.std(ddof=1))


def ladder_cells(arr, axis, protocol):
    """{(variant, method, level): cell}, with §14.5's gate on the candidate protocol."""
    out = {k: c for k, c in arr.items() if k[2] == X.FULL or k[2] in X.AXES[axis]["levels"]}
    if protocol == "candidate":
        read = RP.REGISTERED_READ[axis]
        gated = sorted({m for (_, m, lv) in out if lv != X.FULL and m not in read})
        for k in list(out):
            if k[1] in gated and k[2] != X.FULL:
                out[k] = dict(GATED_CELL)
        if gated:
            print(f"note: candidate {axis} ladder levels NOT shown for {', '.join(gated)} "
                  "(the replication's registered reading not made yet, §14.5)")
    return out


def plot_ladder(lc, axis, protocol):
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    series = [m for m, _ in METHODS]
    for ax, v in zip(axes.ravel(), X.VARIANTS):
        pos = x_positions(axis, v)
        order = sorted(pos, key=lambda lv: pos[lv][0])
        xs_sorted = [pos[lv][0] for lv in order]
        step = 0.5 * min(b - a for a, b in zip(xs_sorted, xs_sorted[1:])) / len(series)
        drawn = False
        for j, m in enumerate(series):
            dx = (j - (len(series) - 1) / 2) * step
            xs, ys, es = [], [], []
            for lv in order:
                c = lc.get((v, m, lv))
                ok = R.cell_ready(c) is None
                mu, sd = stats(c) if ok else (np.nan, np.nan)
                xs.append(pos[lv][0] + dx)
                ys.append(mu)
                es.append(sd)
            if all(np.isnan(ys)):
                continue
            drawn = True
            ax.plot(xs, ys, "-o", color=COLORS[m], lw=1.4, ms=5, label=LABELS[m])
            ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor=COLORS[m], capsize=3, lw=1)
        o = full.get((v, "task_reward")) if protocol == "candidate" else None
        if o is not None and R.cell_ready(o) is None:
            mu, sd = stats(o)
            ax.axhline(mu, ls="--", color=COLORS["task_reward"], lw=1.2, label=LABELS["task_reward"])
            drawn = True
        ax.set_xticks([pos[lv][0] for lv in order])
        ax.set_xticklabels([pos[lv][1] for lv in order])
        ax.set_xlabel("preference pairs" if axis == "sparsity" else "flip fraction p")
        ax.set_ylabel("score (last-10 mean × 100)")
        ax.set_title(f"antmaze-{v}")
        ax.grid(alpha=0.3)
        if not drawn:
            ax.text(0.5, 0.5, "no complete cells yet", ha="center", va="center",
                    transform=ax.transAxes, color="gray")
    handles = {}
    for ax in axes.ravel():
        for h, lab in zip(*ax.get_legend_handles_labels()):
            handles.setdefault(lab, h)
    if handles:
        fig.legend(list(handles.values()), list(handles), loc="lower center", ncol=len(handles), frameon=False)
    fig.suptitle(f"Replication, {axis}, {'WITHOUT' if protocol == 'candidate' else 'with'} terminals "
                 "(mean ± sd over seeds 101-110)")
    fig.tight_layout(rect=(0, 0.05, 1, 0.96))
    plt.show()


for axis in ("sparsity", "noise"):
    plot_ladder(ladder_cells(std, axis, "standard"), axis, "standard")
    plot_ladder(ladder_cells(cand, axis, "candidate"), axis, "candidate")

## RQ1 on the replication (registered)

`rq1_readout.run` and `report`, unchanged, on the replication's cells and accuracies (as
`repl_readout.py rq1`). It prints NOT READ, family by family, until every cell a reading needs is
complete.

In [ ]:
for axis in ("sparsity", "noise"):
    res, use, uwhy, beside = Q.run(axis, std, cand, full, ctrl, acc)
    Q.report(axis, res, use, uwhy, beside,
             ["BESIDE items are not re-run in the replication (§14.83 decision 5): they print as NOT READ"])
    print()